**PADDLEOCR fine tune recognition model (2025/12/21) using real world dataset** epoch 100 and lr 0.0001

**Step 1: Mount Google Drive and Install PaddlecOCR**

In [1]:
!nvidia-smi

Sun Dec 21 06:04:19 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             47W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [4]:
!python -m pip install paddlepaddle-gpu==3.2.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu126/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 8.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.4/201.4 MB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 16.9 MB/s eta 0:00:00
  Attempting uninstall: nvidia-cusparselt-cu12
    Found existing installation: nvidia-cusparselt-cu12 0.7.1
    Uninstalling nvidia-cusparselt-cu12-0.7.1:
      Successfully uninstalled nvidia-cusparselt-cu12-0.7.1
  Attempting uninstall: opt_einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0
  Attempting uninstall: nvidia-nccl-cu12
    Found existing i

In [5]:
!git clone -b release/3.0 https://github.com/PaddlePaddle/PaddleOCR.git
%cd PaddleOCR

Cloning into 'PaddleOCR'...
remote: Enumerating objects: 306564, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 306564 (delta 33), reused 29 (delta 29), pack-reused 306523 (from 2)
Receiving objects: 100% (306564/306564), 1.62 GiB | 40.68 MiB/s, done.
Resolving deltas: 100% (242542/242542), done.
/content/PaddleOCR


In [6]:
# Install Compatible PyTorch
!pip uninstall torch torchvision torchaudio -y
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu118

Found existing installation: torch 2.9.0+cu126
Uninstalling torch-2.9.0+cu126:
  Successfully uninstalled torch-2.9.0+cu126
Found existing installation: torchvision 0.24.0+cu126
Uninstalling torchvision-0.24.0+cu126:
  Successfully uninstalled torchvision-0.24.0+cu126
Found existing installation: torchaudio 2.9.0+cu126
Uninstalling torchaudio-2.9.0+cu126:
  Successfully uninstalled torchaudio-2.9.0+cu126
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 839.6/839.6 MB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 96.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 13.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 82.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 63.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 93.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.5/

In [7]:
!pip install -r requirements.txt
!pip install paddleocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.4/299.4 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 82.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.0/87.0 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/

In [8]:
# Install compatible langchain and restart runtime
!pip install "langchain<0.2.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 129.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 4.5 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.1.2
    Uninstalling tenacity-9.1.2:
      Successfully uninstalled tenacity-9.1.2
  Attempting uninstall: packaging
    Found existing installation: packaging 25.0
    Uninstalling packaging-25.0:
      Successfully uninstalled packaging-25.0
  Attempting uninstall: numpy
    Found existin

In [1]:
# Ensure paddleocr is successfully installed
import paddle
paddle.utils.run_check()
print("CUDA:", paddle.version.cuda())  # 12.6
print("GPU compiled:", paddle.is_compiled_with_cuda())  # True
print("Device:", paddle.device.get_device())  # gpu:0
import torch
print("Torch CUDA:", torch.cuda.is_available())  # True
print("NCCL version:", torch.cuda.nccl.version())  # ~2.27.5

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Running verify PaddlePaddle program ... 


/usr/local/lib/python3.12/dist-packages/paddle/pir/math_op_patch.py:219: UserWarning: Value do not have 'place' interface for pir graph mode, try not to use it. None will be returned.
  warnings.warn(


PaddlePaddle works well on 1 GPU.
PaddlePaddle is installed successfully! Let's start deep learning with PaddlePaddle now.
CUDA: 12.6
GPU compiled: True
Device: gpu:0
Torch CUDA: True
NCCL version: (2, 20, 5)


**Step 2: Import the Dataset**

In [2]:
!ln -s /content/drive/MyDrive/RealWorldDataset /content/PaddleOCR/invoice_dataset

In [3]:
!head /content/PaddleOCR/invoice_dataset/train_labels.txt
!tail /content/PaddleOCR/invoice_dataset/train_labels.txt

20251120_180319_text_0.jpg	THANK YOU!
20251120_180319_text_1.jpg	打印次数：1
20251120_180319_text_2.jpg	打印时間：2025-10-27 17:13:34
20251120_180319_text_3.jpg	【UNPAID】
20251120_180319_text_4.jpg	58.00
20251120_180319_text_5.jpg	應付：
20251120_180319_text_6.jpg	58.00
20251120_180319_text_7.jpg	小計：
20251120_180319_text_8.jpg	58.00
20251120_180319_text_9.jpg	B2厚切三文鱼生井
xcvbhnjmk_text_18.jpg	Remaining Value:
xcvbhnjmk_text_19.jpg	$345.5
xcvbhnjmk_text_20.jpg	Octopus Card 
xcvbhnjmk_text_21.jpg	 No.:
xcvbhnjmk_text_22.jpg	93134609
xcvbhnjmk_text_23.jpg	Redemption Time:
xcvbhnjmk_text_24.jpg	2025-11-1513:35:54
xcvbhnjmk_text_25.jpg	Free Parking
xcvbhnjmk_text_26.jpg	Redemption Receipt
xcvbhnjmk_text_27.jpg	Panda Place


**Step 3: Download the dictionary file and pretrianed model**

In [4]:
!wget -O /content/ppocrv5_dict.txt \
  https://raw.githubusercontent.com/PaddlePaddle/PaddleOCR/main/ppocr/utils/dict/ppocrv5_dict.txt

# Verify
!wc -l /content/ppocrv5_dict.txt

# Preview
!head -n 10 /content/ppocrv5_dict.txt
!tail -n 10 /content/ppocrv5_dict.txt

--2025-12-21 06:14:42--  https://raw.githubusercontent.com/PaddlePaddle/PaddleOCR/main/ppocr/utils/dict/ppocrv5_dict.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 74012 (72K) [text/plain]
Saving to: ‘/content/ppocrv5_dict.txt’

/content/ppocrv5_di 100%[===================>]  72.28K  --.-KB/s    in 0.01s   

2025-12-21 06:14:42 (4.94 MB/s) - ‘/content/ppocrv5_dict.txt’ saved [74012/74012]

18383 /content/ppocrv5_dict.txt
　
一
乙
二
十
丁
厂
七
卜
八
🕞
🕟
🕠
🕡
🕢
🕣
🕤
🕥
🕦
🕧


In [5]:
!mkdir -p /content/pretrain_models
!wget -P /content/pretrain_models/ https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_rec_pretrained.pdparams

--2025-12-21 06:14:50--  https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv5_server_rec_pretrained.pdparams
Resolving paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)... 103.235.47.176, 2402:2b40:7000:628:0:ff:b0e8:88da
Connecting to paddle-model-ecology.bj.bcebos.com (paddle-model-ecology.bj.bcebos.com)|103.235.47.176|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 214594738 (205M) [application/octet-stream]
Saving to: ‘/content/pretrain_models/PP-OCRv5_server_rec_pretrained.pdparams’

PP-OCRv5_server_rec 100%[===================>] 204.65M  12.0MB/s    in 36s     

2025-12-21 06:15:28 (5.68 MB/s) - ‘/content/pretrain_models/PP-OCRv5_server_rec_pretrained.pdparams’ saved [214594738/214594738]



**Optional: Download the config file for editing**

In [6]:
import shutil
shutil.copy('/content/PaddleOCR/configs/rec/PP-OCRv5/PP-OCRv5_server_rec.yml', '/content/ch_invoice_rec.yml')

'/content/ch_invoice_rec.yml'

**Optional: Freeze the backbone of pretrained model and create a frozen model**

In [ ]:
%cd /content/PaddleOCR

/content/PaddleOCR


In [ ]:
import paddle
import yaml
from ppocr.modeling.architectures import build_model

with open('/content/ch_invoice_rec.yml', 'r') as f:
    cfg = yaml.safe_load(f)

model = build_model(cfg['Architecture'])
state_dict = paddle.load('/content/pretrain_models/PP-OCRv5_server_rec_pretrained.pdparams')  # Now valid file
model.set_state_dict(state_dict)

frozen_count = 0
for name, param in model.named_parameters():
    if 'backbone' in name.lower():
        param.stop_gradient = True
        frozen_count += 1
        print(f"Froze: {name}")

print(f"Froze {frozen_count} layers. Trainable params: {sum(p.numel() for p in model.parameters() if not p.stop_gradient)}")

paddle.save(model.state_dict(), '/content/frozen_model.pdparams')

[2025/12/08 13:04:07] ppocr WARNING: Skipping import of the encryption module.


/content/PaddleOCR/ppocr/modeling/heads/rec_nrtr_head.py:371: SyntaxWarning: invalid escape sequence '\d'
  \text{MultiHead}(Q, K, V) = \text{Concat}(head_1,\dots,head_h)W^O


TypeError: MultiHead.__init__() missing 1 required positional argument: 'out_channels_list'

**Step 4: Edit the config file ch_invoice_rec.yml, specify frozen_model or pretrained model for training, then upload in Colab**

In [7]:
%cd /content

/content


In [8]:
from google.colab import files

print("Please select a file to upload:")
uploaded = files.upload()

if not uploaded:
    print("Alert: No file was selected for upload. Please ensure you choose a file.")
else:
    for filename in uploaded.keys():
        print(f"Alert: File '{filename}' has been successfully uploaded.")


Please select a file to upload:


Saving ch_invoice_rec.yml to ch_invoice_rec.yml
Alert: File 'ch_invoice_rec.yml' has been successfully uploaded.


**Step 5: Start Training and Export the inference model**

In [9]:
%cd /content/PaddleOCR

/content/PaddleOCR


In [10]:
!python tools/train.py -c /content/ch_invoice_rec.yml -o Global.use_gpu=True

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[2025/12/21 06:18:05] ppocr WARNING: Skipping import of the encryption module.
[2025/12/21 06:18:05] ppocr INFO: Architecture : 
[2025/12/21 06:18:05] ppocr INFO:     Backbone : 
[2025/12/21 06:18:05] ppocr INFO:         name : PPHGNetV2_B4
[2025/12/21 06:18:05] ppocr INFO:         text_rec : True
[2025/12/21 06:18:05] ppocr INFO:     Head : 
[2025/12/21 06:18:05] ppocr INFO:         head_list : 
[2025/12/21 06:18:05] ppocr INFO:             CTCHead : 
[2025/12/21 06:18:05] ppocr INFO:                 Head : 
[2025/12/21 06:18:05] ppocr INFO:                     fc_decay : 1e-05
[2025/12/21 06:18:05] ppocr INFO:                 Neck : 
[2025/12/21 06:18:05] ppocr INFO:

In [14]:
!python tools/export_model.py -c /content/ch_invoice_rec.yml -o Global.pretrained_model=/content/output/invoice_rec/best_accuracy Global.save_inference_dir=/content/invoice_rec_inference

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
[2025/12/21 07:15:14] ppocr WARNING: Skipping import of the encryption module.
W1221 07:15:16.342073 36700 gpu_resources.cc:114] Please NOTE: device: 0, GPU Compute Capability: 8.0, Driver API Version: 12.4, Runtime API Version: 12.6
[2025/12/21 07:15:17] ppocr INFO: load pretrain successful from /content/output/invoice_rec/best_accuracy
[2025/12/21 07:15:17] ppocr INFO: Export inference config file to /content/invoice_rec_inference/inference.yml
Skipping import of the encryption module
W1221 07:15:19.412225 36700 eager_utils.cc:3441] Paddle static graph(PIR) not support input out tensor for now!!!!!
[2025/12/21 07:15:20] ppocr INFO: inference model is saved to /conten

In [15]:
#Do not copy invoice_rec if not enough disk space
#!cp -r /content/output/invoice_rec /content/drive/MyDrive/
!cp -r /content/invoice_rec_inference /content/drive/MyDrive/

**Step 6: Run Inference using fine-tuned recognition model stored in the folder invoice_rec_inference**

In [16]:
!mkdir -p ./inference_results ./output/det_rec

In [17]:
!paddleocr ocr -i /content/drive/MyDrive/test_images --text_recognition_model_dir /content/invoice_rec_inference --device gpu --save_path ./inference_results --lang ch --ocr_version PP-OCRv5

Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.
/usr/local/lib/python3.12/dist-packages/paddleocr/_utils/cli.py:62: UserWarning: `lang` and `ocr_version` will be ignored when model names or model directories are not `None`.
  wrapper = wrapper_cls(**init_params)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/root/.paddlex/official_models/PP-LCNet_x1_0_doc_ori`.
/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory man

**Step 7: Copy Inference Result**

In [18]:
!cp -r /content/PaddleOCR/inference_results /content/drive/MyDrive/